In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import pickle

from notebooks.imports import *

### Config

In [3]:
from config import dir_config

raw_dir = Path(dir_config.data.raw)
processed_dir = Path(dir_config.data.processed)

processed_dir = Path("/mnt/data/processed")

In [4]:
data = pd.read_csv(Path(processed_dir, "valid_data.csv"))

### Concat motion energy data

In [6]:
import os
import pickle
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

motion_data_dir = Path(processed_dir, "motion_data_dict")
progress_interval = 100  # Print every N files


def load_pickle(file_path):
	try:
		if os.path.getsize(file_path) == 0:
			return None
		with open(file_path, "rb") as f:
			trial_dict = pickle.load(f)
		trial_name = Path(file_path).stem.rsplit("me_array_", 1)[-1]
		return trial_name, trial_dict
	except Exception:
		return None


# Gather all .pkl files
all_files = list(motion_data_dir.glob("*.pkl"))
motion_data_dict = {}

# Thread-based execution for better I/O performance
completed = 0
print(f"🚀 Starting to load {len(all_files)} files...")

with ThreadPoolExecutor() as executor:
	futures = {executor.submit(load_pickle, file): file for file in all_files}
	for future in as_completed(futures):
		result = future.result()
		if result:
			trial_name, trial_dict = result
			motion_data_dict[trial_name] = trial_dict

		completed += 1
		if completed % progress_interval == 0:
			print(f"✅ Loaded {completed}/{len(all_files)} files...")

print(f"🎉 Done! Successfully loaded {len(motion_data_dict)} trial dicts.")


🚀 Starting to load 2 files...
🎉 Done! Successfully loaded 2 trial dicts.


In [8]:
import numpy as np
import torch

# Initialize dictionaries for motion energies
me_global = {}
me_center = {}
me_q1 = {}
me_q2 = {}
me_q3 = {}
me_q4 = {}


# Helper function to handle conversion of tensors to numpy arrays
def convert_to_numpy(value):
	return np.array(value.cpu()) if isinstance(value, torch.Tensor) else value


# Loop over all trial data
for trial_name, trial_dict in motion_data_dict.items():
	# Extract and convert motion energy data
	me_global[trial_name] = convert_to_numpy(trial_dict["global"])
	me_center[trial_name] = convert_to_numpy(trial_dict["center"])

	# Loop over quadrants (Q1 to Q4)
	for quadrant in ["Q1", "Q2", "Q3", "Q4"]:
		locals()[f"me_{quadrant.lower()}"][trial_name] = convert_to_numpy(trial_dict["quadrants"][quadrant])

# save the motion energies
with open(Path(processed_dir, "motion_energies/me_global.pkl"), "wb") as f:
	pickle.dump(me_global, f)
with open(Path(processed_dir, "motion_energies/me_center.pkl"), "wb") as f:
	pickle.dump(me_center, f)
with open(Path(processed_dir, "motion_energies/me_q1.pkl"), "wb") as f:
	pickle.dump(me_q1, f)
with open(Path(processed_dir, "motion_energies/me_q2.pkl"), "wb") as f:
	pickle.dump(me_q2, f)
with open(Path(processed_dir, "motion_energies/me_q3.pkl"), "wb") as f:
	pickle.dump(me_q3, f)
with open(Path(processed_dir, "motion_energies/me_q4.pkl"), "wb") as f:
	pickle.dump(me_q4, f)
print("Motion energies saved successfully.")


KeyError: 'global'

## Load data for normalization

In [8]:
with open(Path(processed_dir, "motion_energies/me_global.pkl"), "rb") as f:
    me_global = pickle.load(f)
with open(Path(processed_dir, "motion_energies/me_center.pkl"), "rb") as f:
    me_center = pickle.load(f)
with open(Path(processed_dir, "motion_energies/me_q1.pkl"), "rb") as f:
    me_q1 = pickle.load(f)
with open(Path(processed_dir, "motion_energies/me_q2.pkl"), "rb") as f:
    me_q2 = pickle.load(f)
with open(Path(processed_dir, "motion_energies/me_q3.pkl"), "rb") as f:
    me_q3 = pickle.load(f)
with open(Path(processed_dir, "motion_energies/me_q4.pkl"), "rb") as f:
    me_q4 = pickle.load(f)

print("Motion energies loaded successfully.")

Motion energies loaded successfully.


In [11]:
# Define output path
output_dir = Path(processed_dir, "motion_energies_normalized")
output_dir.mkdir(parents=True, exist_ok=True)


def pad_motion_energy(me, pad_length=600):
    """Pads motion energy to a fixed length with NaNs."""
    me = np.asarray(me[:-3]).flatten()
    padded = np.full(pad_length, np.nan)
    padded[: min(len(me), pad_length)] = me[:pad_length]
    return padded


def normalize_session(session_data):
    stack = np.vstack(session_data)
    max_val = np.nanmax(np.abs(stack))
    if max_val > 0:
        stack /= max_val
    return stack


# Initialize new dicts to store normalized values
me_global_norm, me_center_norm = {}, {}
me_q1_norm, me_q2_norm, me_q3_norm, me_q4_norm = {}, {}, {}, {}

# Process session-wise
for session_id in data.idx_session.unique():
    session_df = data[data.idx_session == session_id]
    session_codes = session_df.agg_code.values

    # Collect and pad all trials in this session
    global_session, center_session = [], []
    q1_session, q2_session, q3_session, q4_session = [], [], [], []

    for code in session_codes:
        global_session.append(pad_motion_energy(me_global[code]))
        center_session.append(pad_motion_energy(me_center[code]))
        q1_session.append(pad_motion_energy(me_q1[code]))
        q2_session.append(pad_motion_energy(me_q2[code]))
        q3_session.append(pad_motion_energy(me_q3[code]))
        q4_session.append(pad_motion_energy(me_q4[code]))

    # Normalize all motion energies in the session
    global_session = normalize_session(global_session)
    center_session = normalize_session(center_session)
    q1_session = normalize_session(q1_session)
    q2_session = normalize_session(q2_session)
    q3_session = normalize_session(q3_session)
    q4_session = normalize_session(q4_session)

    # Store normalized data
    for i, code in enumerate(session_codes):
        me_global_norm[code] = global_session[i]
        me_center_norm[code] = center_session[i]
        me_q1_norm[code] = q1_session[i]
        me_q2_norm[code] = q2_session[i]
        me_q3_norm[code] = q3_session[i]
        me_q4_norm[code] = q4_session[i]

# Save the normalized dictionaries
with open(output_dir / "me_global.pkl", "wb") as f:
    pickle.dump(me_global_norm, f)
with open(output_dir / "me_center.pkl", "wb") as f:
    pickle.dump(me_center_norm, f)
with open(output_dir / "me_q1.pkl", "wb") as f:
    pickle.dump(me_q1_norm, f)
with open(output_dir / "me_q2.pkl", "wb") as f:
    pickle.dump(me_q2_norm, f)
with open(output_dir / "me_q3.pkl", "wb") as f:
    pickle.dump(me_q3_norm, f)
with open(output_dir / "me_q4.pkl", "wb") as f:
    pickle.dump(me_q4_norm, f)

print("✅ All normalized motion energy files saved.")

✅ All normalized motion energy files saved.
